# Logits Preprocessing and Data Engineering

In [ ]:
def default_params(): 
    return {
        'current_model': 'M1', 
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/semeru-datasets/code_smells/transformation',
            'transformation': 'curated',
            'content_column': 'code',
            'sampling_size': 500,
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'output_path' : '/workspaces/CodeSmells/datax/code_smells/logits',
        'callbacks_path' : '/workspaces/CodeSmells/datax/code_smells/callbacks',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc

In [3]:
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

In [4]:
from transformers import CodeLlamaTokenizer, LlamaForCausalLM
from datasets import load_dataset

In [5]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [6]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}/{params['dataset']['transformation']}"
create_folder(log_file)
log_file += '/data_en.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [7]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### Dataset

In [8]:
print(f"{params['dataset']['path']}/{params['dataset']['transformation']}_{params['dataset']['sampling_size']}.json")

/workspaces/CodeSmells/semeru-datasets/code_smells/extraction/curated_500.json


In [10]:
df_dataset = pd.read_json(f"{params['dataset']['path']}/{params['dataset']['transformation']}_{params['dataset']['sampling_size']}.json", )

In [11]:
df_dataset

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,n_ast_nodes,n_identifiers,s_msg_id,s_line,s_column,s_end_line,s_end_column,s_code,category,input_lenght
0,256261,a59bca366174d9c692fa19750c24d65f47660ef7,haystack,haystack/modeling/training/base.py,base.py,_get_state_dict,Apply black formatting (#2115)\n\n* Testing bl...,def _get_state_dict(self):\n \n ...,https://github.com/deepset-ai/haystack.git,Python,...,193,20,W0311,2,0,2,22,state_dict = {,Warning,296
1,305801,6f564e4f514b56bce281ec7e82703cfbff87b417,core,homeassistant/components/ring/binary_sensor.py,binary_sensor.py,async_added_to_hass,Improve entity type hints [r] (#77874),async def async_added_to_hass(self) -> None:\n...,https://github.com/home-assistant/core.git,Python,...,63,6,W0311,4,0,4,37,self._dings_update_callback(),Warning,73
2,70649,5fe901e5d86ed02dbbb63039a897582951266afd,wagtail,wagtail/admin/tests/pages/test_edit_page.py,test_edit_page.py,test_new_comment,Fix commenting thread notifications being sent...,def test_new_comment(self):\n post_data...,https://github.com/wagtail/wagtail.git,Python,...,565,37,C0301,33,0,33,125,self.assertEqual(mail.outbox[0].subjec...,Convention,666
3,151753,bdfedb5fcb02b88c600ef25c88bbb5d939b8bd0a,freqtrade,freqtrade/freqai/RL/BaseReinforcementLearningM...,BaseReinforcementLearningModel.py,__init__,Improve typehints / reduce warnings from mypy,"def __init__(self, **kwargs) -> None:\n ...",https://github.com/freqtrade/freqtrade.git,Python,...,411,38,W0311,13,0,13,44,import_str = 'stable_baselines3',Warning,494
4,3868,2282a4ae0221b1fb88e16eca8bc14a166998d2d2,airbyte,airbyte-integrations/connectors/source-hubspot...,streams.py,state,🎉 Source Hubspot: Migrate to CDK (#10177)\n\n*...,"def state(self, value):\n state_value =...",https://github.com/airbytehq/airbyte.git,Python,...,99,14,W0311,7,0,7,61,"self._start_date = max(self._state, se...",Warning,110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1702546,43471,09f38ad3f6872bae5059a1de226362eb358c4a7a,airflow,tests/providers/microsoft/azure/operators/test...,test_asb.py,test_send_message_queue,Implement Azure Service Bus Queue Operators (#...,"def test_send_message_queue(self, mock_get_con...",https://github.com/apache/airflow.git,Python,...,132,21,C2801,10,12,13,24,mock.call()\n .__enter__()\n ...,Convention,193
1704801,196984,4a6d5d342e1d0111130d4b31708535b862bfacd0,sympy,sympy/printing/repr.py,repr.py,_print_Permutation,Update the deprecation for Permutation.print_c...,"def _print_Permutation(self, expr):\n f...",https://github.com/sympy/sympy.git,Python,...,388,30,C2801,20,16,20,53,Cycle(expr)(expr.size - 1).__repr__(),Convention,441
1705975,105,0b8a53bd313abdf484a9d1e3fbd6aad13c0ec857,PySyft,packages/syft/tests/syft/core/tensor/passthrou...,passthrough_test.py,test__rshift__,adding tests,def test__rshift__() -> None:\n data_a = np...,https://github.com/OpenMined/PySyft.git,Python,...,183,17,C2801,7,15,7,44,tensor_a.__rshift__(tensor_b),Convention,204
1717859,117464,9ce5a21dd6359fd7e8ebf78051ce9e97bd195ec9,mindsdb,tests/unit/executor_test_base.py,executor_test_base.py,set_executor,ML handler supbrocess (#3377)\n\n* log -> logg...,"def set_executor(self, to_mock_model_controlle...",https://github.com/mindsdb/mindsdb.git,Python,...,422,53,C2801,49,27,49,51,config_patch.__enter__(),Convention,610


#### Model Loading

In [38]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = CodeLlamaTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     if params['quantization'] == 'int4':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
     elif params['quantization'] == 'int8':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
     elif params['quantization'] == 'float32':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
     elif params['quantization'] == 'float16':
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
     else: 
          model = LlamaForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [39]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [40]:
model.config

LlamaConfig {
  "_name_or_path": "codellama/CodeLlama-7b-hf",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_position_embeddings": 16384,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.36.2",
  "use_cache": true,
  "vocab_size": 32016
}

#### preprocess dataset

In [56]:
df_dataset['input_ids'] = df_dataset[params['dataset']['content_column']].map(lambda code: tokenizer.encode(code, add_special_tokens=False))
df_dataset['input_lenght'] = df_dataset['input_ids'].map(lambda input_ids: len(input_ids))

#### Softmax Normalization and Data Engineering

In [57]:
def topk_tuple( logit_vocab_tensor, largest, tokenizer_fn):
    "Run topk for a token"
    topk = logit_vocab_tensor.topk( k=1 , largest=largest ) #TODO K number of elements can be extended
    return ( tokenizer.convert_tokens_to_string([tokenizer_fn.decode(topk.indices)]), topk.values.item())

def min_max_logits( logit_vocab_sample_tensor, tokenizer_fn ):
    "Compute min_max for a sample"
    max_cases = []
    min_cases = []
    for logit_vocab_tensor in logit_vocab_sample_tensor:
        max_cases.append( topk_tuple( logit_vocab_tensor = logit_vocab_tensor, largest = True, tokenizer_fn = tokenizer_fn) ) #TST Max Logit
        min_cases.append( topk_tuple( logit_vocab_tensor = logit_vocab_tensor, largest = False, tokenizer_fn = tokenizer_fn) ) #TST Min Logit
    return max_cases, min_cases

def actual_logit( 
                 logit_vocab_sample_tensor, 
                 tokenized_prompt, 
                 tokenizer_fn,
                 ):
    "Compute actual logits for a sample"
    actual_logits_prompt = []
    for token_pos, id_token in enumerate( tokenized_prompt[1:] ): #Eliminate the first token prediction since we do not use it
        actual_logits_prompt.append(
            (   tokenizer.convert_tokens_to_string([tokenizer_fn.decode( int(id_token))]), #retrieving the name of the token with the id
                logit_vocab_sample_tensor[token_pos][int(id_token)].item()) #retrieving the logit given the position in the sequence and the position in the vocab
            )
    return actual_logits_prompt

In [58]:
soft = torch.nn.Softmax( dim = 0 ) #Flattening normalization

In [59]:
callbacks_dir = f"{params['callbacks_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}"
out = np.load(f"{callbacks_dir}/logits_tensor[0]_batch[0].npy")

print(out.shape) #<sample,tokens,voc_tokens>
out = out[0]


(1, 296, 32016)


In [60]:
max_case,min_case = min_max_logits(
    logit_vocab_sample_tensor = [ soft( torch.from_numpy(token) ) for token in out], ####### 
    tokenizer_fn= tokenizer
    )
print(max_case)
assert len(max_case) == len(min_case)

[('<PRE>', 0.7492886185646057), ('module', 0.47944238781929016), ('get', 0.12872232496738434), ('_', 0.8901668787002563), ('data', 0.010642001405358315), ('_', 0.462878942489624), ('dict', 0.09639009833335876), ('(', 0.7244718670845032), ('model', 0.20590771734714508), ('):', 0.513270914554596), ('\n', 0.9693825244903564), ('      ', 0.8319970965385437), ('\n', 0.8800777196884155), ('      ', 0.9487285017967224), ('state', 0.29825857281684875), ('_', 0.8827518224716187), ('dict', 0.9903534650802612), ('=', 0.9614717960357666), ('{}', 0.27781277894973755), ('\n', 0.8859555125236511), ('          ', 0.83038729429245), ("'", 0.6328372359275818), ('model', 0.05148650333285332), ('ation', 0.49834975600242615), ('_', 0.5450510382652283), ('on', 0.0955599918961525), ('":', 0.7114249467849731), ('self', 0.9179317951202393), ('.', 0.8560599684715271), ('evalu', 0.8842658996582031), ('ate', 0.9936426877975464), ('_', 0.9982263445854187), ('every', 0.9977918863296509), (',', 0.9776225686073303), 

In [64]:
assert tokenizer.decode(df_dataset['input_ids'][0]) == df_dataset[params['dataset']['content_column']][0]
df_dataset[params['dataset']['content_column']][0]

'def _get_state_dict(self):\n        \n        state_dict = {\n            "evaluate_every": self.evaluate_every,\n            "n_gpu": self.n_gpu,\n            "grad_acc_steps": self.grad_acc_steps,\n            "device": self.device,\n            "local_rank": self.local_rank,\n            "early_stopping": self.early_stopping,\n            "epochs": self.epochs,\n            "checkpoint_on_sigterm": self.checkpoint_on_sigterm,\n            "checkpoint_root_dir": self.checkpoint_root_dir,\n            "checkpoint_every": self.checkpoint_every,\n            "checkpoints_to_keep": self.checkpoints_to_keep,\n            "from_epoch": self.from_epoch,\n            "from_step": self.from_step,\n            "global_step": self.global_step,\n            "log_learning_rate": self.log_learning_rate,\n            "log_loss_every": self.log_loss_every,\n            "disable_tqdm": self.disable_tqdm,\n        }\n\n        return state_dict\n'

In [65]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

actual_cases = actual_logit(
    logit_vocab_sample_tensor = [ soft( torch.from_numpy(token) ) for token in out] , #Out is a complete sequence
    tokenized_prompt = input_ids_list[0], ## SAMPLE ID
    tokenizer_fn = tokenizer
    )
actual_cases

[('def', 0.0007611338514834642),
 ('_', 0.009010307490825653),
 ('get', 0.12872232496738434),
 ('_', 0.8901668787002563),
 ('state', 0.0015619436744600534),
 ('_', 0.462878942489624),
 ('dict', 0.09639009833335876),
 ('(', 0.7244718670845032),
 ('self', 0.19837789237499237),
 ('):', 0.513270914554596),
 ('\n', 0.9693825244903564),
 ('       ', 0.0029633562080562115),
 ('\n', 0.8800777196884155),
 ('      ', 0.9487285017967224),
 ('state', 0.29825857281684875),
 ('_', 0.8827518224716187),
 ('dict', 0.9903534650802612),
 ('=', 0.9614717960357666),
 ('{', 0.25638824701309204),
 ('\n', 0.8859555125236511),
 ('          ', 0.83038729429245),
 ('"', 0.2546458840370178),
 ('evalu', 0.0002774437016341835),
 ('ate', 0.1546919345855713),
 ('_', 0.5450510382652283),
 ('every', 0.08610284328460693),
 ('":', 0.7114249467849731),
 ('self', 0.9179317951202393),
 ('.', 0.8560599684715271),
 ('evalu', 0.8842658996582031),
 ('ate', 0.9936426877975464),
 ('_', 0.9982263445854187),
 ('every', 0.9977918863

#### Processing all the Batches

In [66]:
def batching_logits(tokenizer,tf_input_ids,size=10000):
    max_logit_token_prompt = []
    min_logit_token_prompt = []
    actual_logit_token_prompt = []

    
    soft = torch.nn.Softmax( dim = 0 )                          #Flattening normalization
    
    for file in range( size ):
        callbacks_dir = f"{params['callbacks_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}"
        out = np.load(f"{callbacks_dir}/logits_tensor[{file}]_batch[{file}].npy") #<sample,tokens,voc_tokens>
        out = out[0]  ##### #<tokens,voc_tokens>
        next_tokens_distribution = [ soft( torch.from_numpy(token) ) for token in out]  #Flattening normalization
        
        max_cases,min_cases = min_max_logits(
            logit_vocab_sample_tensor = next_tokens_distribution,
            tokenizer_fn= tokenizer
            )

        actual_cases = actual_logit(
            logit_vocab_sample_tensor = next_tokens_distribution,
            tokenized_prompt = tf_input_ids[ file ],
            tokenizer_fn = tokenizer
            )
        
        max_logit_token_prompt.append( max_cases )
        min_logit_token_prompt.append( min_cases )
        actual_logit_token_prompt.append( actual_cases )
        
        logging.info(file)
    return max_logit_token_prompt,min_logit_token_prompt,actual_logit_token_prompt

In [67]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

In [68]:
max_logit_token_prompt, min_logit_token_prompt, actual_logit_token_prompt = batching_logits(
    tokenizer=tokenizer , tf_input_ids=input_ids_list, 
    size = len(df_dataset)
) #<---WARNING TIME Consuming

#### Saving results

In [69]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [70]:
dataframe_to_save = df_dataset.copy()
dataframe_to_save['max_prob'] = max_logit_token_prompt
dataframe_to_save['min_prob'] = min_logit_token_prompt
dataframe_to_save['actual_prob'] = actual_logit_token_prompt
dataframe_to_save.shape

(5, 33)

In [72]:
output_dir = f"{params['output_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}"
create_folder(output_dir)
dataframe_to_save.to_json(f"{output_dir}/raw_logits.json", index=False)

#### Loss Retrieval

In [75]:
def batching_loss( size = dataframe_to_save.shape[0] ):
    output_dir = f"{params['callbacks_path']}/{params['current_model']}_q_{params['quantization']}/{params['dataset']['transformation']}"
    output_loss = []
    for current_batch in range(size):
        out = np.load(f"{output_dir}/_loss_batch[{current_batch}].npy")
        output_loss.append( out.item() ) #.item() for numpy library
        logging.info(current_batch)
    return output_loss

In [76]:
output_loss = batching_loss() #[WAENING!] Takes Time

In [77]:
output_loss

[0.6501871347427368,
 1.36393404006958,
 0.6265979409217834,
 0.9561377763748169,
 1.176944613456726]

In [78]:
dataframe_to_save['loss'] = output_loss
dataframe_to_save.head(5)

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,s_end_line,s_end_column,s_code,category,input_lenght,input_ids,max_prob,min_prob,actual_prob,loss
0,256261,a59bca366174d9c692fa19750c24d65f47660ef7,haystack,haystack/modeling/training/base.py,base.py,_get_state_dict,Apply black formatting (#2115)\n\n* Testing bl...,def _get_state_dict(self):\n \n ...,https://github.com/deepset-ai/haystack.git,Python,...,2,22,state_dict = {,Warning,295,"[822, 903, 657, 29918, 3859, 29918, 8977, 2989...","[(<PRE>, 0.7492886185646057), (module, 0.47944...","[(<s>, 6.321602312800434e-13), ($}, 1.21012630...","[(def, 0.0007611338514834642), (_, 0.009010307...",0.650187
1,305801,6f564e4f514b56bce281ec7e82703cfbff87b417,core,homeassistant/components/ring/binary_sensor.py,binary_sensor.py,async_added_to_hass,Improve entity type hints [r] (#77874),async def async_added_to_hass(self) -> None:\n...,https://github.com/home-assistant/core.git,Python,...,4,37,self._dings_update_callback(),Warning,72,"[7465, 822, 7465, 29918, 23959, 29918, 517, 29...","[(<PRE>, 0.7492926716804504), (\n, 0.145589932...","[(<s>, 6.322408417115677e-13), (oreferrer, 3.7...","[(async, 1.0811768333951477e-05), (def, 0.0006...",1.363934
2,70649,5fe901e5d86ed02dbbb63039a897582951266afd,wagtail,wagtail/admin/tests/pages/test_edit_page.py,test_edit_page.py,test_new_comment,Fix commenting thread notifications being sent...,def test_new_comment(self):\n post_data...,https://github.com/wagtail/wagtail.git,Python,...,33,125,self.assertEqual(mail.outbox[0].subjec...,Convention,665,"[822, 1243, 29918, 1482, 29918, 9342, 29898, 1...","[(<PRE>, 0.7492899298667908), (module, 0.47944...","[(<s>, 6.321468413832132e-13), ($}, 1.21012033...","[(def, 0.0007611322216689587), (test, 0.019638...",0.626598
3,151753,bdfedb5fcb02b88c600ef25c88bbb5d939b8bd0a,freqtrade,freqtrade/freqai/RL/BaseReinforcementLearningM...,BaseReinforcementLearningModel.py,__init__,Improve typehints / reduce warnings from mypy,"def __init__(self, **kwargs) -> None:\n ...",https://github.com/freqtrade/freqtrade.git,Python,...,13,44,import_str = 'stable_baselines3',Warning,493,"[822, 4770, 2344, 12035, 1311, 29892, 3579, 19...","[(<PRE>, 0.7492900490760803), (module, 0.47944...","[(<s>, 6.321457571810407e-13), ($}, 1.21011436...","[(def, 0.0007611316395923495), (__, 0.00598208...",0.956138
4,3868,2282a4ae0221b1fb88e16eca8bc14a166998d2d2,airbyte,airbyte-integrations/connectors/source-hubspot...,streams.py,state,🎉 Source Hubspot: Migrate to CDK (#10177)\n\n*...,"def state(self, value):\n state_value =...",https://github.com/airbytehq/airbyte.git,Python,...,7,61,"self._start_date = max(self._state, se...",Warning,109,"[822, 2106, 29898, 1311, 29892, 995, 1125, 13,...","[(<PRE>, 0.749289870262146), (module, 0.479445...","[(<s>, 6.321889626376143e-13), ($}, 1.21010756...","[(def, 0.0007611394976265728), (state, 6.87381...",1.176945


In [79]:
## Saving CheckPoint 2
dataframe_to_save.to_json(f"{output_dir}/raw_logits.json", index=False)

In [80]:
print("================================= PROCESS COMPLETED =================================")

================================= PROCESS COMPLETE =================================


In [82]:
del model
torch.cuda.empty_cache()
gc.collect()

0